# YeastCaduceus — Phase 2: MLM Domain Adaptation

**Goal:** Align PlantCAD2-Small embeddings to fungal sequence distribution via masked language modeling on the 165_Saccharomycetales corpus.

**Run order (every session):**
1. Cell 1 — Mount Drive + install
2. Cell 2 — Paths + constants
3. Cell 3 — Model + LoRA init *(NUCLEOTIDE_IDS fixed here — uses tokenizer() call not convert_tokens_to_ids)*
4. Cell 4 — PEFT forward patch *(must run every session — never re-run without full restart)*
5. Cell 5 — **ONE-TIME** pre-tokenize → parquet *(skip if parquet already on Drive)*
6. Cell 6 — Load pre-tokenized parquet from Drive → RAM
7. Cell 7 — Collator smoke-check
8. Cell 8 — Training *(auto-resumes from checkpoint)*
9. Cell 9 — Val perplexity + save adapter

**Key constraints:**
- `CaduceusForMaskedLM.forward()` accepts **only** `input_ids` (pure SSM — no attention mechanism)
- Gradient checkpointing unimplemented in `modeling_caduceus.py` (`# TODO` in source)
- PEFT patch: must be at `model.forward` (PeftModel instance level), NOT `base_model.model.forward`
- Never re-run Cell 4 in the same session — re-patching `__func__` causes RecursionError
- Drive parquet: ~30s load into RAM vs minutes of `shutil.copytree` over FUSE


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Mount Drive + install
# Version pins are non-negotiable. Do NOT change them.
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Step 1 — PyTorch pinned to cu121 (mamba-ssm prebuilt wheel ABI requirement)
!pip3 install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121 -q

# Step 2 — mamba-ssm against transformers==4.40.0 (temporary pin)
!pip3 install mamba-ssm==2.2.2 transformers==4.40.0 -q
# sentence-transformers conflict warning here is benign

# Step 3 — causal-conv1d: separate Mamba2 CUDA kernel
# Omitting causes AttributeError: 'NoneType'.causal_conv1d_fwd at runtime (not install time)
!pip3 install causal-conv1d==1.4.0 -q

# Step 4 — upgrade transformers + peft now that mamba compiled
!pip3 install transformers==4.46.3 peft==0.14.0 -q

!pip3 install datasets biopython pandas numpy tqdm -q

import subprocess
expected = {
    'torch':         '2.3.1+cu121',
    'mamba-ssm':     '2.2.2',
    'causal-conv1d': '1.4.0',
    'transformers':  '4.46.3',
    'peft':          '0.14.0',
}
print('Package versions:')
all_ok = True
for pkg, exp in expected.items():
    result = subprocess.run(['pip', 'show', pkg], capture_output=True, text=True)
    ver = next((l.split(': ')[1] for l in result.stdout.splitlines()
                if l.startswith('Version')), 'NOT FOUND')
    ok = '✅' if exp in ver else '❌'
    if '❌' in ok:
        all_ok = False
    print(f'  {ok} {pkg}: {ver}')

assert all_ok, 'Version mismatch — re-run Cell 1'
print('\n✅ Cell 1 done')


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 62.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 117.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 117.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 63.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 150.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 21.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 49.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.3 MB/s eta 0:00:00
     

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Paths, constants
# ─────────────────────────────────────────────────────────────────────────────

import torch
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE            = Path('/content/drive/MyDrive/yeastcaduceus')
MLM_DATASET     = BASE / 'data/mlm_dataset'                 # raw dataset (sequence + is_repeat)
PARQUET_TRAIN   = BASE / 'data/mlm_tokenized_train.parquet' # pre-tokenized, single file
PARQUET_VAL     = BASE / 'data/mlm_tokenized_val.parquet'
PARQUET_TEST    = BASE / 'data/mlm_tokenized_test.parquet'
CHECKPOINT_DIR  = BASE / 'checkpoints/phase2'
RESULTS_DIR     = BASE / 'results'
MODEL_ID        = 'kuleshov-group/PlantCAD2-Small-l24-d0768'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Training constants (PlantCAD2 paper Methods + Kuleshov pre-training CLI) ──
WINDOW_SIZE          = 8192
PHYSICAL_BATCH_SIZE  = 2       # OOM at batch=8 — gradient checkpointing not implemented in Caduceus
GRAD_ACCUM_STEPS     = 64      # effective batch = 2 × 64 = 128  (matches paper)
LEARNING_RATE        = 1e-4
WARMUP_STEPS         = 50
NUM_EPOCHS           = 3
MASK_PROB            = 0.15    # BERT-style MLM
REPEAT_WEIGHT        = 0.1     # soft-masked (repeat) positions weighted at 10%
LOG_STEPS            = 50

# ── LoRA config (from Kuleshov HF adapter repos + PlantCAD2 paper Methods) ───
LORA_R               = 8
LORA_ALPHA           = 32
LORA_DROPOUT         = 0.1
LORA_TARGET_MODULES  = ['out_proj', 'x_proj', 'in_proj']

# ── GPU info ──────────────────────────────────────────────────────────────────
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Effective batch size: {PHYSICAL_BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {PHYSICAL_BATCH_SIZE * GRAD_ACCUM_STEPS}')
print('\n✅ Cell 2 done')


GPU  : NVIDIA A100-SXM4-40GB
VRAM : 42.4 GB
Effective batch size: 2 x 64 = 128

✅ Cell 2 done


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Model + LoRA init
#
# NUCLEOTIDE_IDS: must use tokenizer() call, NOT convert_tokens_to_ids().
#   convert_tokens_to_ids('A') returns unk_token_id for all keys in the
#   Caduceus vocab → NUCLEOTIDE_IDS = [] → collator crashes later with
#   ValueError: 'a' cannot be empty unless no samples are taken.
#   Fix: tokenize the character string, extract input_ids[0].
#   Confirmed: A=3, C=4, G=5, T=6.
#
# task_type=FEATURE_EXTRACTION: no classification head — LoRA trained on MLM loss.
# Gradient checkpointing will fail gracefully (# TODO in modeling_caduceus.py).
# ─────────────────────────────────────────────────────────────────────────────

import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from peft import LoraConfig, get_peft_model, TaskType

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

MASK_TOKEN_ID = tokenizer.mask_token_id


# ── NUCLEOTIDE_IDS: tokenize directly, never use convert_tokens_to_ids ────────
_nuc_map = {}
for nuc in ['A', 'C', 'G', 'T']:
    ids = tokenizer(nuc, add_special_tokens=False)['input_ids']
    assert ids, f'Tokenizer returned empty ids for {nuc}'
    _nuc_map[nuc] = ids[0]
NUCLEOTIDE_IDS = list(_nuc_map.values())
assert len(NUCLEOTIDE_IDS) == 4, f'Expected 4 IDs, got: {NUCLEOTIDE_IDS}'
print(f'  Nucleotide IDs: A={_nuc_map["A"]}, C={_nuc_map["C"]}, G={_nuc_map["G"]}, T={_nuc_map["T"]}')

# ── Model ─────────────────────────────────────────────────────────────────────
print('\nLoading PlantCAD2-Small...')
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

# ── LoRA ──────────────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias='none',
    task_type=TaskType.FEATURE_EXTRACTION,
)
model = get_peft_model(model, lora_config)

# ── VOCAB_SIZE: derive from model, NOT tokenizer ───────────────────────────
# tokenizer.vocab_size = 7, but lm_head output dim = 8.
# The Caduceus authors added an 8th output neuron without updating the
# tokenizer's reported vocab_size. Always derive from the actual layer.
VOCAB_SIZE = model.base_model.model.lm_head.weight.shape[0]
print(f'  VOCAB_SIZE (from lm_head): {VOCAB_SIZE}')   # 8
print(f'  [MASK] token id          : {MASK_TOKEN_ID}')

# Gradient checkpointing — will fail with informative message (# TODO in source)
try:
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable()
    print('Gradient checkpointing: enabled')
except Exception as e:
    print(f'Gradient checkpointing: not available ({e})')
    print('  Reduce PHYSICAL_BATCH_SIZE to 1 if OOM')

model = model.to('cuda:0')

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nParameters:')
print(f'  Total     : {total_params / 1e6:.1f}M')
print(f'  Trainable : {trainable_params / 1e6:.2f}M ({100 * trainable_params / total_params:.2f}%)')
print(f'  Frozen    : {(total_params - trainable_params) / 1e6:.1f}M')
print(f'  VRAM after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

assert trainable_params / total_params < 0.05, 'Too many trainable params — check LoRA config'
print('\n✅ Cell 3 done')


Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/785 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/419 [00:00<?, ?B/s]

...

modeling_caduceus.py: 0.00B [00:00, ?B/s]

modeling_rcps.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/kuleshov-group/PlantCAD2-Small-l24-d0768:
- modeling_rcps.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/kuleshov-group/PlantCAD2-Small-l24-d0768:
- modeling_caduceus.py
- modeling_rcps.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/704M [00:00<?, ?B/s]

  VOCAB_SIZE (from lm_head): 8
  [MASK] token id          : 1
Gradient checkpointing: not available (CaduceusForMaskedLM does not support gradient checkpointing.)
  Reduce PHYSICAL_BATCH_SIZE to 1 if OOM

Parameters:
  Total     : 178.4M
  Trainable : 2.42M (1.36%)
  Frozen    : 176.0M
  VRAM after load: 0.71 GB

✅ Cell 3 done


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — PEFT forward patch
#
# WHY THIS IS NEEDED:
#   CaduceusForMaskedLM.forward() accepts ONLY input_ids. Caduceus is a pure
#   SSM — attention_mask is architecturally meaningless (no attention matrix).
#   PEFT's PeftModelForFeatureExtraction.forward() has attention_mask,
#   output_attentions, output_hidden_states, return_dict, task_ids hardcoded
#   in its own signature and passes them explicitly to self.base_model() even
#   as None — this crashes Caduceus with TypeError regardless of dict pops in
#   compute_loss.
#
# WHY model.forward (PeftModel level), NOT model.base_model.model.forward:
#   Patching at Caduceus level uses __func__ which captures whatever .forward
#   currently is. Re-running in the same session causes __func__ to capture
#   the already-patched version -> infinite RecursionError. Patching at the
#   PeftModel instance level avoids this: the patch is a new bound method
#   referencing the original base_model directly.
#
# SANITY CHECK:
#   A dummy forward pass runs at the end to confirm no crash.
#
# !! NEVER RE-RUN THIS CELL IN THE SAME SESSION !!
#    If you need to re-run it: Runtime -> Restart session, then re-run all cells.
# ─────────────────────────────────────────────────────────────────────────────

import types

def _caduceus_peft_forward(
    self,
    input_ids=None,
    attention_mask=None,        # PEFT hardcodes these in its signature — absorb and drop
    inputs_embeds=None,
    output_attentions=None,
    output_hidden_states=None,
    return_dict=None,
    task_ids=None,
    **kwargs,
):
    # _enable_peft_forward_hooks is required for LoRA adapter routing
    with self._enable_peft_forward_hooks(**kwargs):
        kwargs = {k: v for k, v in kwargs.items() if k not in self.special_peft_forward_args}
        return self.base_model(input_ids=input_ids)  # only input_ids reaches Caduceus

model.forward = types.MethodType(_caduceus_peft_forward, model)
print('PEFT forward patched — only input_ids forwarded to CaduceusForMaskedLM')

# Sanity: confirm forward pass does not crash
import torch
_dummy = torch.randint(1, 7, (1, 32), device='cuda')
with torch.no_grad():
    _out = model(_dummy)
assert hasattr(_out, 'logits'), 'Forward pass returned unexpected output type'
assert _out.logits.shape == (1, 32, VOCAB_SIZE), f'Unexpected logits shape: {_out.logits.shape}'
del _dummy, _out
print('Forward pass sanity check passed — shape (1, 32, {}) OK'.format(VOCAB_SIZE))
print('\n✅ Cell 4 done')


PEFT forward patched — only input_ids forwarded to CaduceusForMaskedLM
Forward pass sanity check passed — shape (1, 32, 8) OK

✅ Cell 4 done


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — ONE-TIME: pre-tokenize dataset -> save as parquet on Drive
#
# WHY PARQUET instead of save_to_disk (Arrow directory):
#   save_to_disk() creates hundreds of small Arrow shard files. Copying them
#   to local SSD each session uses shutil.copytree() which makes one FUSE/HTTPS
#   round-trip per file (~50-200ms each over the Drive FUSE mount) -> minutes
#   of I/O before training starts.
#
#   A single parquet file = one sequential read over FUSE (~30s) then fully
#   loaded into RAM via Dataset.from_parquet(). All DataLoader reads are from
#   RAM during training — zero Drive I/O.
#
# WHY PRE-TOKENIZE:
#   Caduceus tokenizer uses trust_remote_code=True and custom Python. In
#   DataLoader workers it is pickled/unpickled and called on 8192-char strings
#   every batch -> major bottleneck (observed: 0.02 it/s). Pre-tokenizing
#   eliminates the tokenizer from the DataLoader hot path entirely. The collator
#   only does masking (pure tensor ops).
#
# Parquet stores input_ids (list of 8192 ints) and is_repeat (list of 8192
# bools) per row. Only these two fields are needed for training.
#
# This cell is idempotent — skips if parquet files already exist on Drive.
# ─────────────────────────────────────────────────────────────────────────────

from datasets import load_from_disk

if PARQUET_TRAIN.exists() and PARQUET_VAL.exists():
    print('Pre-tokenized parquet already exists on Drive — skipping Cell 5')
    print(f'  Train : {PARQUET_TRAIN}')
    print(f'  Val   : {PARQUET_VAL}')
else:
    print('Loading raw MLM dataset from Drive...')
    ds_train_raw = load_from_disk(str(MLM_DATASET / 'train'))
    ds_val_raw   = load_from_disk(str(MLM_DATASET / 'val'))
    print(f'  train: {len(ds_train_raw):,} | val: {len(ds_val_raw):,}')
    assert 'is_repeat' in ds_train_raw.features,         '"is_repeat" missing — run Phase 1 Cell 2c first'
    assert 'sequence' in ds_train_raw.features,         '"sequence" field missing from raw dataset'

    def tokenize_batch(batch):
        encoded = tokenizer(
            batch['sequence'],
            add_special_tokens=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            padding=False,
            truncation=False,
        )
        return {'input_ids': encoded['input_ids']}

    KEEP = ('input_ids', 'is_repeat')

    print('Tokenizing train (~5-10 min)...')
    ds_train_tok = ds_train_raw.map(
        tokenize_batch, batched=True, batch_size=512, num_proc=2,
        desc='Tokenizing train',
        remove_columns=[c for c in ds_train_raw.column_names if c not in KEEP],
    )

    print('Tokenizing val...')
    ds_val_tok = ds_val_raw.map(
        tokenize_batch, batched=True, batch_size=512, num_proc=2,
        desc='Tokenizing val',
        remove_columns=[c for c in ds_val_raw.column_names if c not in KEEP],
    )

    # Also tokenize test split while we have the tokenizer loaded
    print('Tokenizing test...')
    ds_test_raw = load_from_disk(str(MLM_DATASET / 'test'))
    ds_test_tok = ds_test_raw.map(
        tokenize_batch, batched=True, batch_size=512, num_proc=2,
        desc='Tokenizing test',
        remove_columns=[c for c in ds_test_raw.column_names if c not in KEEP],
    )

    # Verify token length
    assert len(ds_train_tok[0]['input_ids']) == WINDOW_SIZE,         f'Token length mismatch: {len(ds_train_tok[0]["input_ids"])} != {WINDOW_SIZE}'
    assert len(ds_train_tok[0]['is_repeat']) == WINDOW_SIZE,         f'is_repeat length mismatch: {len(ds_train_tok[0]["is_repeat"])} != {WINDOW_SIZE}'

    print('Saving to Drive as parquet...')
    ds_train_tok.to_parquet(str(PARQUET_TRAIN))
    ds_val_tok.to_parquet(str(PARQUET_VAL))
    ds_test_tok.to_parquet(str(PARQUET_TEST))

    print(f'  Train : {PARQUET_TRAIN.stat().st_size / 1e9:.2f} GB')
    print(f'  Val   : {PARQUET_VAL.stat().st_size / 1e6:.1f} MB')
    print(f'  Test  : {PARQUET_TEST.stat().st_size / 1e6:.1f} MB')
    print('\n✅ Cell 5 done — parquet saved to Drive')


Pre-tokenized parquet already exists on Drive — skipping Cell 5
  Train : /content/drive/MyDrive/yeastcaduceus/data/mlm_tokenized_train.parquet
  Val   : /content/drive/MyDrive/yeastcaduceus/data/mlm_tokenized_val.parquet


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Load pre-tokenized parquet from Drive -> RAM
#
# from_parquet() reads the single file sequentially over FUSE (~30s) and loads
# the dataset fully into Colab RAM (~83 GB on A100 instance). All subsequent
# DataLoader reads come from RAM — zero Drive I/O during training.
# ─────────────────────────────────────────────────────────────────────────────

from datasets import Dataset
import time

assert PARQUET_TRAIN.exists(),     f'Parquet not found: {PARQUET_TRAIN} — run Cell 5 first'

print('Loading train parquet from Drive -> RAM...')
t0 = time.time()
ds_train_tok = Dataset.from_parquet(str(PARQUET_TRAIN))
print(f'  Loaded in {time.time()-t0:.0f}s : {len(ds_train_tok):,} windows')

print('Loading val parquet from Drive -> RAM...')
t0 = time.time()
ds_val_tok = Dataset.from_parquet(str(PARQUET_VAL))
print(f'  Loaded in {time.time()-t0:.0f}s : {len(ds_val_tok):,} windows')

print(f'\nFeatures: {list(ds_train_tok.features.keys())}')

# Verify shapes
sample = ds_train_tok[0]
assert len(sample['input_ids']) == WINDOW_SIZE,     f'input_ids length {len(sample["input_ids"])} != {WINDOW_SIZE}'
assert len(sample['is_repeat']) == WINDOW_SIZE,     f'is_repeat length {len(sample["is_repeat"])} != {WINDOW_SIZE}'
print(f'  input_ids length : {len(sample["input_ids"])}')
print(f'  is_repeat length : {len(sample["is_repeat"])}')

steps_per_epoch = len(ds_train_tok) // (PHYSICAL_BATCH_SIZE * GRAD_ACCUM_STEPS)
print(f'\nSteps/epoch: ~{steps_per_epoch:,}')
print('\n✅ Cell 6 done')


Loading train parquet from Drive -> RAM...


Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/57 [00:00<?, ?it/s]

  Loaded in 98s : 846,664 windows
Loading val parquet from Drive -> RAM...


Generating train split: 0 examples [00:00, ? examples/s]

  Loaded in 5s : 1,121 windows

Features: ['is_repeat', 'input_ids']
  input_ids length : 8192
  is_repeat length : 8192

Steps/epoch: ~6,614

✅ Cell 6 done


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Custom MLM collator + smoke-check
#
# Uses pre-tokenized input_ids directly — tokenizer NOT called here.
# Workers only do masking (pure tensor ops) -> fast in DataLoader hot path.
#
# BERT masking strategy:
#   80% of masked positions -> replace with [MASK]
#   10% of masked positions -> replace with random nucleotide (A/C/G/T)
#   10% of masked positions -> keep unchanged (but included in labels for loss)
#
# Returns:
#   input_ids : [B, 8192]  masked token ids
#   labels    : [B, 8192]  original ids at masked positions, -100 elsewhere
#   is_repeat : [B, 8192]  per-position repeat mask (float) for loss weighting
#
# Does NOT return attention_mask — Caduceus forward() rejects it.
# ─────────────────────────────────────────────────────────────────────────────

import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, List

@dataclass
class CaduceusMLMCollator:
    mask_prob:      float
    nucleotide_ids: List[int]
    mask_token_id:  int

    def __post_init__(self):
        assert len(self.nucleotide_ids) == 4, 'NUCLEOTIDE_IDS must have 4 entries'

    def __call__(self, batch: List[Dict]) -> Dict[str, torch.Tensor]:
        # Use pre-tokenized input_ids — no tokenizer call
        input_ids  = torch.tensor([item['input_ids']  for item in batch], dtype=torch.long)
        is_repeats = torch.tensor([item['is_repeat']  for item in batch], dtype=torch.float)
        B, L = input_ids.shape

        # Clone for labels before any masking
        labels = input_ids.clone()

        # Sample 15% of positions to mask
        masked_positions = torch.bernoulli(torch.full((B, L), self.mask_prob)).bool()

        # -100 at non-masked positions (CrossEntropyLoss ignore_index)
        labels[~masked_positions] = -100

        # Split masked positions: 80% MASK, 10% random, 10% unchanged
        rand           = torch.rand(B, L)
        mask_indices   = masked_positions & (rand < 0.80)
        random_indices = masked_positions & (rand >= 0.80) & (rand < 0.90)

        input_ids[mask_indices] = self.mask_token_id

        if random_indices.any():
            n = random_indices.sum().item()
            input_ids[random_indices] = torch.tensor(
                np.random.choice(self.nucleotide_ids, size=n), dtype=torch.long
            )

        return {
            'input_ids': input_ids,    # [B, 8192]
            'labels':    labels,       # [B, 8192]
            'is_repeat': is_repeats,   # [B, 8192]
        }


collator = CaduceusMLMCollator(
    mask_prob=MASK_PROB,
    nucleotide_ids=NUCLEOTIDE_IDS,
    mask_token_id=MASK_TOKEN_ID,
)

# ── Smoke-check ───────────────────────────────────────────────────────────────
sample_batch = [ds_train_tok[i] for i in range(4)]
out = collator(sample_batch)

print('Collator smoke-check:')
print(f'  input_ids : {out["input_ids"].shape}  dtype={out["input_ids"].dtype}')
print(f'  labels    : {out["labels"].shape}')
print(f'  is_repeat : {out["is_repeat"].shape}')

frac = (out['labels'] != -100).float().mean().item()
print(f'  Masked frac: {frac:.3f}  (expected ~{MASK_PROB})')

assert abs(frac - MASK_PROB) < 0.02, f'Masking fraction off: {frac:.3f}'
assert 'attention_mask' not in out, 'attention_mask in batch — will crash Caduceus'
assert out['input_ids'].shape == (4, WINDOW_SIZE)
assert out['is_repeat'].shape == (4, WINDOW_SIZE)

print('\n✅ Cell 7 done')


Collator smoke-check:
  input_ids : torch.Size([4, 8192])  dtype=torch.int64
  labels    : torch.Size([4, 8192])
  is_repeat : torch.Size([4, 8192])
  Masked frac: 0.151  (expected ~0.15)

✅ Cell 7 done


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7.5 — Baseline perplexity (frozen PlantCAD2-Small, BEFORE adaptation)
#
# Run ONCE before Cell 8. Records the starting point for the paper:
#   "PlantCAD2-Small achieves perplexity X on Saccharomycetales sequences
#    before domain adaptation, reduced to Y after 3 epochs of MLM."
#
# Uses same collator and val set as training — directly comparable numbers.
# ─────────────────────────────────────────────────────────────────────────────

import math, json, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

model.eval()
baseline_loader = DataLoader(
    ds_val_tok,
    batch_size=PHYSICAL_BATCH_SIZE,
    collate_fn=collator,
    num_workers=2,
    pin_memory=True,
)

total_loss, total_tokens = 0.0, 0

with torch.no_grad():
    for batch in baseline_loader:
        input_ids = batch['input_ids'].to('cuda')
        labels    = batch['labels'].to('cuda')
        is_repeat = batch['is_repeat'].to('cuda')

        outputs = model(input_ids=input_ids)
        logits  = outputs.logits          # [B, L, V]
        B, L, V = logits.shape

        per_token = F.cross_entropy(
            logits.view(-1, V), labels.view(-1),
            reduction='none', ignore_index=-100,
        ).view(B, L)

        repeat_weights = 1.0 - (1.0 - REPEAT_WEIGHT) * is_repeat
        active_mask    = (labels != -100).float()
        weighted_loss  = (per_token * repeat_weights * active_mask).sum()
        denom          = (repeat_weights * active_mask).sum()

        total_loss   += weighted_loss.item()
        total_tokens += denom.item()

baseline_loss = total_loss / total_tokens
baseline_ppl  = math.exp(baseline_loss)

print(f'Baseline val loss       : {baseline_loss:.4f}')
print(f'Baseline val perplexity : {baseline_ppl:.2f}')

# Save to results CSV as epoch 0
import pandas as pd
baseline_df = pd.DataFrame([{
    'epoch': 0, 'eval_loss': baseline_loss, 'perplexity': baseline_ppl
}])
metrics_path = RESULTS_DIR / 'phase2_mlm_perplexity.csv'
baseline_df.to_csv(metrics_path, index=False)
print(f'Baseline saved to: {metrics_path}')

# Also write into training_meta at the end — store this now for Cell 9
BASELINE_PPL = baseline_ppl   # global, picked up by Cell 9 meta dict

model.train()
print('\n✅ Cell 7.5 done — proceed to Cell 8 (training)')

Baseline val loss       : 1.2194
Baseline val perplexity : 3.39
Baseline saved to: /content/drive/MyDrive/yeastcaduceus/results/phase2_mlm_perplexity.csv

✅ Cell 7.5 done — proceed to Cell 8 (training)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Training
#
# RepeatAwareMLMTrainer.compute_loss:
#   1. Pops attention_mask (Trainer injects it; SSM rejects it)
#   2. Pops is_repeat (not a model input — used for loss weighting only)
#   3. Per-token CE loss (reduction='none')
#   4. Repeat down-weighting: weight = 1.0 - (1 - REPEAT_WEIGHT) * is_repeat
#      -> 1.0 at normal positions, 0.1 at soft-masked repeat positions
#   5. Mean over masked positions only (labels != -100)
#
# prediction_loss_only=True: eval routes through compute_loss — no
#   prediction_step override needed. Matches Kuleshov HF_pre_train.py CLI.
#
# save_strategy='epoch' and eval_strategy='epoch' must match:
#   load_best_model_at_end=True requires both to use the same unit.
#
# Session drop recovery:
#   Re-run Cells 1 -> 2 -> 3 -> 4 -> 6 -> 7 -> 8
#   Trainer auto-resumes from latest checkpoint in CHECKPOINT_DIR.
# ─────────────────────────────────────────────────────────────────────────────

import os
import torch
import torch.nn as nn
from transformers import Trainer, TrainingArguments


class RepeatAwareMLMTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        inputs.pop('attention_mask', None)                       # SSM rejects this
        is_repeat = inputs.pop('is_repeat').to(model.device)    # [B, L] float
        labels    = inputs.pop('labels').to(model.device)       # [B, L]

        outputs = model(**inputs)                                # only input_ids
        logits  = outputs.logits                                # [B, L, V]
        B, L, V = logits.shape

        loss_fct = nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
        per_token_loss = loss_fct(
            logits.view(-1, V), labels.view(-1)
        ).view(B, L)

        # 1.0 at non-repeat, REPEAT_WEIGHT at soft-masked positions
        repeat_weights = 1.0 - (1.0 - REPEAT_WEIGHT) * is_repeat
        active_mask    = (labels != -100).float()

        loss = (per_token_loss * repeat_weights * active_mask).sum() /                (repeat_weights * active_mask).sum().clamp(min=1e-8)

        return (loss, outputs) if return_outputs else loss


# ── Resume detection ──────────────────────────────────────────────────────────
checkpoints = sorted([
    d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('checkpoint-')
]) if CHECKPOINT_DIR.exists() else []
resume_from = str(CHECKPOINT_DIR / checkpoints[-1]) if checkpoints else None
print('Resuming from:', checkpoints[-1] if resume_from else 'scratch')

# ── Training arguments ────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    # Batch size — effective = 2 x 64 = 128 (PlantCAD2 paper)
    per_device_train_batch_size=PHYSICAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    per_device_eval_batch_size=PHYSICAL_BATCH_SIZE,

    # Precision
    bf16=True,
    bf16_full_eval=True,

    # Routes eval through compute_loss (required for SSM compatibility)
    # Matches official Kuleshov HF_pre_train.py --prediction_loss_only True
    prediction_loss_only=True,

    # Optimiser — fused single-kernel AdamW, ~10% step speedup on A100
    optim='adamw_torch_fused',
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='linear',
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,

    # Epochs + checkpointing
    # save_strategy and eval_strategy MUST match for load_best_model_at_end=True
    num_train_epochs=NUM_EPOCHS,
    save_strategy='epoch',
    save_total_limit=3,
    load_best_model_at_end=True,

    eval_strategy='epoch',
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # Logging
    logging_steps=LOG_STEPS,
    logging_dir=str(RESULTS_DIR / 'phase2_logs'),
    report_to='wandb',

    # DataLoader — dataset in RAM, workers do masking only (no tokenizer)
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    remove_unused_columns=False,   # must preserve is_repeat field in batch
)

trainer = RepeatAwareMLMTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_train_tok,
    eval_dataset=ds_val_tok,
    data_collator=collator,
)

# ── Train ─────────────────────────────────────────────────────────────────────
print('Starting MLM domain adaptation...')
print(f'  train : {len(ds_train_tok):,} windows')
print(f'  val   : {len(ds_val_tok):,} windows')
print(f'  batch : {PHYSICAL_BATCH_SIZE} physical x {GRAD_ACCUM_STEPS} accum = {PHYSICAL_BATCH_SIZE * GRAD_ACCUM_STEPS} effective')
print(f'  epochs: {NUM_EPOCHS}')
print()

train_result = trainer.train(resume_from_checkpoint=resume_from)

print('\nTraining complete.')
print(f'  Final train loss: {train_result.training_loss:.4f}')
print('\n✅ Cell 8 done')


Resuming from: scratch
Starting MLM domain adaptation...
  train : 846,664 windows
  val   : 1,121 windows
  batch : 2 physical x 64 accum = 128 effective
  epochs: 3



Epoch,Training Loss,Validation Loss


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Validation perplexity + save adapter
# ─────────────────────────────────────────────────────────────────────────────

import math, json
import pandas as pd
from datasets import Dataset

# ── Per-epoch perplexity from trainer log history ─────────────────────────────
eval_rows = [
    {
        'epoch':      e['epoch'],
        'eval_loss':  e['eval_loss'],
        'perplexity': math.exp(e['eval_loss']),
    }
    for e in trainer.state.log_history if 'eval_loss' in e
]

metrics_df = pd.DataFrame(eval_rows)
print('Val perplexity per epoch:')
print(metrics_df.to_string(index=False))

metrics_path = RESULTS_DIR / 'phase2_mlm_perplexity.csv'
metrics_df.to_csv(metrics_path, index=False)
print(f'\nMetrics saved: {metrics_path}')

# ── Final eval on test split ──────────────────────────────────────────────────
assert PARQUET_TEST.exists(), f'Test parquet not found: {PARQUET_TEST}'
print('\nLoading test split...')
ds_test_tok   = Dataset.from_parquet(str(PARQUET_TEST))
test_results  = trainer.evaluate(eval_dataset=ds_test_tok)
test_ppl      = math.exp(test_results['eval_loss'])
print(f'  eval_loss  : {test_results["eval_loss"]:.4f}')
print(f'  perplexity : {test_ppl:.2f}')

pd.concat([metrics_df, pd.DataFrame([{
    'epoch': 'test', 'eval_loss': test_results['eval_loss'], 'perplexity': test_ppl
}])]).to_csv(metrics_path, index=False)

# ── Save LoRA adapter weights ─────────────────────────────────────────────────
adapter_path = BASE / 'checkpoints/phase2/final_adapter'
adapter_path.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))

meta = {
    'base_model':      MODEL_ID,
    'phase':           'domain_adaptation_mlm',
    'corpus':          '165_Saccharomycetales',
    'train_windows':   len(ds_train_tok),
    'val_windows':     len(ds_val_tok),
    'lora_r':          LORA_R,
    'lora_alpha':      LORA_ALPHA,
    'lora_dropout':    LORA_DROPOUT,
    'target_modules':  LORA_TARGET_MODULES,
    'mask_prob':       MASK_PROB,
    'repeat_weight':   REPEAT_WEIGHT,
    'lr':              LEARNING_RATE,
    'effective_batch': PHYSICAL_BATCH_SIZE * GRAD_ACCUM_STEPS,
    'epochs':          NUM_EPOCHS,
    'warmup_steps':    WARMUP_STEPS,
    'final_val_ppl':   round(eval_rows[-1]['perplexity'], 4) if eval_rows else None,
    'final_test_ppl':  round(test_ppl, 4),
    'baseline_ppl': round(BASELINE_PPL, 4),
}
with open(adapter_path / 'training_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f'\nLoRA adapter saved: {adapter_path}')

# ── Go / No-Go ────────────────────────────────────────────────────────────────
print(f'\n{"="*50}')
print('GO / NO-GO')
ppls       = [r['perplexity'] for r in eval_rows]
decreasing = all(ppls[i] >= ppls[i+1] for i in range(len(ppls)-1))
checks = {
    'Val perplexity decreasing across epochs':  decreasing,
    'Adapter weights saved':                    (adapter_path / 'adapter_model.safetensors').exists(),
    'Training metadata saved':                  (adapter_path / 'training_meta.json').exists(),
    'Metrics CSV saved':                        metrics_path.exists(),
}
for check, result in checks.items():
    print(f'  {"✅" if result else "❌"} {check}')

if not decreasing and len(ppls) > 1:
    print('  Perplexity not monotonically decreasing — check lr or LoRA rank')

print(f'\nPhase 2 complete')
print(f'Load adapter in Phase 3:')
print(f'  model = PeftModel.from_pretrained(base_model, "{adapter_path}")')
print('\n✅ Cell 9 done')
